# Iris Classifier using Vertex AI

## Overview

In this tutorial, you build a scikit-learn model and deploy it on infer in local environment using Google Cloud Storage for logging and tracking model and data


### Dataset

This tutorial uses R.A. Fisher's Iris dataset, a small and popular dataset for machine learning experiments. Each instance has four numerical features, which are different measurements of a flower, and a target label that
categorizes the flower into: **Iris setosa**, **Iris versicolour** and **Iris virginica**.

This tutorial uses [a version of the Iris dataset available in the
scikit-learn library](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html#sklearn.datasets.load_iris).

## Installing Dependencies

In [ ]:
# Vertex SDK for Python
%pip install --upgrade --quiet  google-cloud-aiplatform scikit-learn

In [1]:
PROJECT_ID = "project-d5c04e1d-be23-4954-84c"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

BUCKET_URI = f"gs://mlops-course-project-d5c04e1d-be23-4954-84c-week-1"  # @param {type:"string"}
MODEL_ARTIFACT_DIR="iris_classifier/model"

In [ ]:
! gcloud storage buckets create {BUCKET_URI} --location={LOCATION} --project={PROJECT_ID}

## Task-2 Store Data in GCS

In [32]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

BUCKET_URI = "gs://mlops-course-project-d5c04e1d-be23-4954-84c-week-1"

# --- PIPELINE PARAMETER ---
DATA_VERSION = "v0" # Toggle this to "v0", "v1", or "v2" for different runs
# --------------------------

print(f"--- Starting Data Ingestion & Split for {DATA_VERSION} ---")

# 1. Load the raw dataset dynamically
raw_data_path = f'data/{DATA_VERSION}/iris.csv'
print(f"Loading raw dataset from {raw_data_path}...")
data = pd.read_csv(raw_data_path)

# 2. Split into Train and Test (60/40 split)
print("Splitting data into train and test sets...")
train, test_data = train_test_split(data, test_size=0.4, stratify=data['species'], random_state=42)

--- Starting Data Ingestion & Split for v0 ---
Loading raw dataset from data/v0/iris.csv...
Splitting data into train and test sets...


In [33]:
# 3. Save locally to the versioned directory
os.makedirs(f'data/{DATA_VERSION}', exist_ok=True)
local_train_path = f'data/{DATA_VERSION}/train.csv'
local_test_path = f'data/{DATA_VERSION}/test.csv'

train.to_csv(local_train_path, index=False)
test_data.to_csv(local_test_path, index=False)
print(f"Data saved locally to {local_train_path} and {local_test_path}")

# 4. Upload to the versioned GCS bucket path
gcs_data_dir = f"{BUCKET_URI}/data/{DATA_VERSION}"
print(f"Uploading files to {gcs_data_dir}/ ...")
! gcloud storage cp {local_train_path} {gcs_data_dir}/train.csv
! gcloud storage cp {local_test_path} {gcs_data_dir}/test.csv

print(f"\nTask 2 Complete: {DATA_VERSION} data successfully ingested and stored in GCS.")

Data saved locally to data/v0/train.csv and data/v0/test.csv
Uploading files to gs://mlops-course-project-d5c04e1d-be23-4954-84c-week-1/data/v0/ ...
Copying file://data/v0/train.csv to gs://mlops-course-project-d5c04e1d-be23-4954-84c-week-1/data/v0/train.csv
  Completed files 1/1 | 800.0B/800.0B                                          
Copying file://data/v0/test.csv to gs://mlops-course-project-d5c04e1d-be23-4954-84c-week-1/data/v0/test.csv
  Completed files 1/1 | 568.0B/568.0B                                          

Task 2 Complete: v0 data successfully ingested and stored in GCS.


## Task-3 Execute the IRIS Training Pipeline

In [34]:
import pandas as pd
import joblib
import os
from datetime import datetime
from sklearn.tree import DecisionTreeClassifier

BUCKET_URI = "gs://mlops-course-project-d5c04e1d-be23-4954-84c-week-1"

# --- PIPELINE PARAMETER ---
DATA_VERSION = "v0" # Ensure this matches the data you want to train on!
# --------------------------

print(f"--- Starting Training Pipeline on {DATA_VERSION} ---")

# 1. Fetch versioned training data from GCS
gcs_train_path = f"{BUCKET_URI}/data/{DATA_VERSION}/train.csv"
local_dl_path = f"./train_downloaded_{DATA_VERSION}.csv"

print(f"Fetching data from {gcs_train_path}...")
! gcloud storage cp {gcs_train_path} {local_dl_path}

# 2. Load data and separate features/target
train_df = pd.read_csv(local_dl_path)
X_train = train_df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
y_train = train_df['species']

--- Starting Training Pipeline on v0 ---
Fetching data from gs://mlops-course-project-d5c04e1d-be23-4954-84c-week-1/data/v0/train.csv...
Copying gs://mlops-course-project-d5c04e1d-be23-4954-84c-week-1/data/v0/train.csv to file://./train_downloaded_v0.csv
  Completed files 1/1 | 800.0B/800.0B                                          


In [35]:
# 3. Train the Model
print("Training the model...")
mod_dt = DecisionTreeClassifier(max_depth=3, random_state=1)
mod_dt.fit(X_train, y_train)

# 4. Save the model locally
os.makedirs('artifacts', exist_ok=True)
local_model_path = "artifacts/model.joblib"
joblib.dump(mod_dt, local_model_path)

Training the model...


['artifacts/model.joblib']

In [36]:
# 5. Generate Timestamp & Tag with Data Version
timestamp = datetime.now().strftime("%Y-%m-%dT%H-%M-%S")
# Appending the version ensures you know exactly what trained this model
gcs_artifact_uri = f"{BUCKET_URI}/artifacts/{timestamp}_{DATA_VERSION}"

print(f"Uploading model artifact to GCS at: {gcs_artifact_uri}...")
! gcloud storage cp {local_model_path} {gcs_artifact_uri}/model.joblib

print(f"\nTask 3 Complete: Training artifact successfully stored.")

# Save the tagged timestamp so Task 4 grabs the exact right model
LATEST_RUN_TIMESTAMP = f"{timestamp}_{DATA_VERSION}"

Uploading model artifact to GCS at: gs://mlops-course-project-d5c04e1d-be23-4954-84c-week-1/artifacts/2026-06-19T14-01-59_v0...
Copying file://artifacts/model.joblib to gs://mlops-course-project-d5c04e1d-be23-4954-84c-week-1/artifacts/2026-06-19T14-01-59_v0/model.joblib
  Completed files 1/1 | 2.2kiB/2.2kiB                                          

Task 3 Complete: Training artifact successfully stored.


## Task-4 Run Inference on Evaluation Set

In [37]:
import pandas as pd
import joblib
import os
from sklearn import metrics

BUCKET_URI = "gs://mlops-course-project-d5c04e1d-be23-4954-84c-week-1"

# --- PIPELINE PARAMETER ---
DATA_VERSION = "v0" 
# --------------------------

print(f"--- Starting Inference Pipeline for Run: {LATEST_RUN_TIMESTAMP} ---")

# 1. Define version-specific GCS paths 
gcs_model_path = f"{BUCKET_URI}/artifacts/{LATEST_RUN_TIMESTAMP}/model.joblib"
gcs_test_data_path = f"{BUCKET_URI}/data/{DATA_VERSION}/test.csv" 

local_model_path = "./model_downloaded.joblib"
local_test_path = f"./test_downloaded_{DATA_VERSION}.csv"

# 2. Fetch the Artifact and the specific Version's Test Data
print("Fetching model and versioned test data from GCS...")
! gcloud storage cp {gcs_model_path} {local_model_path}
! gcloud storage cp {gcs_test_data_path} {local_test_path}

--- Starting Inference Pipeline for Run: 2026-06-19T14-01-59_v0 ---
Fetching model and versioned test data from GCS...
Copying gs://mlops-course-project-d5c04e1d-be23-4954-84c-week-1/artifacts/2026-06-19T14-01-59_v0/model.joblib to file://./model_downloaded.joblib
  Completed files 1/1 | 2.2kiB/2.2kiB                                          
Copying gs://mlops-course-project-d5c04e1d-be23-4954-84c-week-1/data/v0/test.csv to file://./test_downloaded_v0.csv
  Completed files 1/1 | 568.0B/568.0B                                          


In [38]:
# 3. Load Model and Data into memory
print("Loading model and running predictions...")
loaded_model = joblib.load(local_model_path)
test_df = pd.read_csv(local_test_path)

# Separate features and target 
X_test = test_df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
y_test = test_df['species']

# 4. Run Inference
predictions = loaded_model.predict(X_test)

# 5. Evaluate and Output Metrics
accuracy = metrics.accuracy_score(y_test, predictions)
print("\n--- Inference Results ---")
print(f"The accuracy of the model trained & tested on {DATA_VERSION} data is: {accuracy:.3f}")

Loading model and running predictions...

--- Inference Results ---
The accuracy of the model trained & tested on v0 data is: 1.000
